# Phase 1 — Step 4b: Retrieval demo (see chunk size in action)

**Goal:** make "too small fragments / too large blurs" tangible. Same question, three chunk sizes, side-by-side top-3 hits.

> **⚠️ Throwaway learning notebook — not a template for production shape.**
> Real pipelines split this into two phases:
> - **Ingestion (offline, once per doc):** parse → chunk → embed → write to a persistent vector store.
> - **Retrieval (online, per query):** embed the query → search the existing index → return top-K. No parsing or chunking at query time.
>
> Here both are collapsed into one notebook on purpose: to vary chunk size and *see* its effect on retrieval, we need to re-chunk + re-embed three ways in the same run. In-memory FAISS, no persistence, dies with the kernel. The "real" shapes live in `nb_02_chunk.ipynb` (ingestion: parse + chunk) and upcoming nb_03 (embed + write to vector store) / nb_04 (retrieval-only).

**Pipeline mini (this notebook):**
1. Re-parse + re-chunk the FSR PDF in 3 sizes (small / baseline / large) — *only because we're comparing sizes; real ingestion picks one*.
2. Embed every chunk via the LiteLLM gateway (`azure-text-embedding-3-large-1`).
3. Build a tiny in-memory FAISS index per variant.
4. Embed the question, search top-3, print results.

**What you'll watch for:**
- Small chunks: hits look fragmented; the answer-bearing context may be split across 2 chunks.
- Baseline: hits feel "about the right amount" — a paragraph or two of relevant content.
- Large chunks: hits contain a lot of unrelated material around the relevant bit (dilution); fewer chunks need to be retrieved but each is noisier.

**Note:** we're skipping persistence, batching, error handling, etc. Real Phase 1 step 5/6 notebooks will do this properly.

## 1. Setup — env, parse, chunk

Reusing the same logic from `nb_02_chunk.ipynb`. Self-contained so this notebook runs standalone.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load LiteLLM gateway creds from ai-arch/.env
load_dotenv(Path("/home/u560060992/dbx/ai-arch/.env"))
assert os.getenv("LITELLM_API_KEY"), "LITELLM_API_KEY missing — check .env"
print("gateway:", os.getenv("LITELLM_BASE_URL"))
print("embedding model:", os.getenv("EMBEDDING_MODEL"))

gateway: https://dev-gateway.apps.gevernova.net
embedding model: azure-text-embedding-3-large-1


In [2]:
import fitz  # PyMuPDF
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = Path("/home/u560060992/dbx/ai-arch/sample-docs/fsr-sample-01.pdf")
doc = fitz.open(PDF_PATH)
pages = [{"page": i + 1, "text": doc.load_page(i).get_text()} for i in range(doc.page_count)]
doc.close()
print(f"PDF: {PDF_PATH.name}  →  {len(pages)} pages, {sum(len(p['text']) for p in pages):,} chars total")

enc = tiktoken.get_encoding("cl100k_base")
n_tokens = lambda t: len(enc.encode(t))

def chunk_pages(pages, size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        length_function=n_tokens,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    out = []
    for p in pages:
        for i, piece in enumerate(splitter.split_text(p["text"])):
            out.append({"page": p["page"], "text": piece, "tokens": n_tokens(piece)})
    return out

variants = {
    "small (300/50)":      chunk_pages(pages, 300, 50),
    "baseline (1000/200)": chunk_pages(pages, 1000, 200),
    "large (1500/300)":    chunk_pages(pages, 1500, 300),
}
print(f"\n{len(pages)} pages → chunk counts per strategy:")
for name, ch in variants.items():
    avg_tok = sum(c["tokens"] for c in ch) / len(ch)
    max_tok = max(c["tokens"] for c in ch)
    print(f"  {name:<22} → {len(ch):>4} chunks   (avg {avg_tok:>5.0f} tok, max {max_tok:>4} tok)")
print("\nNote: per-page chunking caps each chunk at one page's worth of text,")
print("so 'large' often produces the same chunk count as 'baseline' on short-page PDFs like FSRs.")

PDF: fsr-sample-01.pdf  →  51 pages, 20,243 chars total

51 pages → chunk counts per strategy:
  small (300/50)         →   54 chunks   (avg   127 tok, max  295 tok)
  baseline (1000/200)    →   51 chunks   (avg   133 tok, max  425 tok)
  large (1500/300)       →   51 chunks   (avg   133 tok, max  425 tok)

Note: per-page chunking caps each chunk at one page's worth of text,
so 'large' often produces the same chunk count as 'baseline' on short-page PDFs like FSRs.


## 2. Embed — call the embedding model (not an LLM)

We call the **embedding model** (`text-embedding-3-large`) through the LiteLLM gateway. Important distinction: the gateway hosts both embedding models and LLMs, but they're different models with different jobs:

- **Embedding model** → reads text, outputs one fixed-size vector (3,072 floats). Used for search/similarity. Cheap.
- **LLM** (e.g. `gemini-3-flash`) → reads text, generates more text. Used for answers. Much more expensive.

We use the OpenAI Python SDK with `base_url` pointed at the gateway because OpenAI's API shape (`/v1/embeddings`) became the de facto standard — Azure, LiteLLM, and most servers copy it, so the same SDK works everywhere.

**What's a vector here?** A list of 3,072 floats. Two pieces of text that mean similar things → vectors pointing in similar directions. Always 3,072 regardless of input length (see Q&A on fixed output dimension).

In [3]:
import httpx
import numpy as np
from openai import OpenAI

verify_ssl = os.getenv("LLM_VERIFY_SSL", "false").lower() == "true"
# trust_env=False → ignore http_proxy/https_proxy env vars (Zscaler).
# The gateway is internal (apps.gevernova.net), reachable directly without the proxy.
client = OpenAI(
    base_url=os.getenv("LITELLM_BASE_URL") + "/v1",
    api_key=os.getenv("LITELLM_API_KEY"),
    http_client=httpx.Client(verify=verify_ssl, trust_env=False),
)
EMBED_MODEL = os.getenv("EMBEDDING_MODEL")

def embed_texts(texts: list[str], batch_size: int = 64) -> np.ndarray:
    """Embed a list of strings; return shape (len(texts), dim) float32 array."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend(d.embedding for d in resp.data)
    arr = np.array(out, dtype="float32")
    return arr

# Smoke test on one short string
v = embed_texts(["hello world"])
print("vector shape:", v.shape, "  dim:", v.shape[1])

vector shape: (1, 3072)   dim: 3072


In [4]:
# Embed all chunks for all 3 variants. This is the slow cell — ~couple minutes.
embeddings = {}
for name, ch in variants.items():
    print(f"embedding {name}: {len(ch)} chunks…")
    embeddings[name] = embed_texts([c["text"] for c in ch])
    print(f"  → shape {embeddings[name].shape}")

embedding small (300/50): 54 chunks…
  → shape (54, 3072)
embedding baseline (1000/200): 51 chunks…
  → shape (51, 3072)
embedding large (1500/300): 51 chunks…
  → shape (51, 3072)


## 3. Index — tiny in-memory FAISS

FAISS = Facebook AI Similarity Search. The local equivalent of "vector store". We use the simplest variant: `IndexFlatIP` (inner-product over normalized vectors = cosine similarity). For ~hundreds of chunks this is instant.

We also **L2-normalize** the vectors so inner-product == cosine similarity. (Convention: cosine is the standard distance for text embeddings.)

In [5]:
import faiss

def build_index(vectors: np.ndarray) -> faiss.Index:
    vecs = vectors.copy()
    faiss.normalize_L2(vecs)             # in-place
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return idx

indexes = {name: build_index(emb) for name, emb in embeddings.items()}
for name, idx in indexes.items():
    print(f"{name:<22} index: {idx.ntotal} vectors, dim {idx.d}")

small (300/50)         index: 54 vectors, dim 3072
baseline (1000/200)    index: 51 vectors, dim 3072
large (1500/300)       index: 51 vectors, dim 3072


## 4. Retrieve — same question, three indexes

For a given question:
1. Embed the question with the **same model** as the chunks (different model = nonsense distances).
2. Normalize it.
3. Ask each index for top-3 nearest chunks.
4. Print short previews side by side.

Pick a question that should have a real answer in the FSR. Edit `QUERY` below to try different phrasings.

In [6]:
QUERY = "What was the equipment serial number and the failure observed?"
TOP_K = 3

q_vec = embed_texts([QUERY])
faiss.normalize_L2(q_vec)

def preview(text: str, n: int = 220) -> str:
    t = " ".join(text.split())  # collapse whitespace
    return t[:n] + ("…" if len(t) > n else "")

for name, idx in indexes.items():
    ch = variants[name]
    scores, ids = idx.search(q_vec, TOP_K)
    print("=" * 80)
    print(f"{name}   query: {QUERY!r}")
    print("=" * 80)
    for rank, (score, i) in enumerate(zip(scores[0], ids[0]), start=1):
        c = ch[i]
        print(f"\n  #{rank}  score={score:.3f}  page={c['page']}  tokens={c['tokens']}")
        print(f"     {preview(c['text'])}")
    print()

small (300/50)   query: 'What was the equipment serial number and the failure observed?'

  #1  score=0.413  page=1  tokens=249
     Unit 14 visual A Inspection MIDLAND COGEN Outage Start Date: 20 Aug 2025 ESN/SY: 810893 | SY0048230 Oracle Project ID: A-1960052 | EV-185177 | EVP-555614 Report Issued: 20 Oct 2025 Prepared By Justin Clark Field Engineer…

  #2  score=0.402  page=5  tokens=270
     GAS TURBINE (810893 | SY0048230) U N I T 1 Summary S E C T I O N 1 . 1 T i m e r s a n d C o u n t e r s 1.1 Timers and Counters F O R M Technology Gas Turbine Total Starts 4982 Operating Hours 196258.9 Equivalent Operat…

  #3  score=0.381  page=36  tokens=170
     2 . 4 . 5 B l a d e a n d v a n e R o w 2 , 3 , a n d 4 2.4.5 Blade and vane Row 2, 3, and 4 F O R M Part Description: Row 2 blades and vanes had minor erosion. Rows 3 and 4 were founded in acceptable condition. 25082100…

baseline (1000/200)   query: 'What was the equipment serial number and the failure observed?'

  #1  score=0.41

## 5. What to look for

Re-run the previous cell with different `QUERY` values. Examples to try:
- `"What was the equipment serial number and the failure observed?"` — multi-fact question (small chunks may split)
- `"customer site location"` — short factual lookup (small often wins)
- `"summarize the recommended next actions"` — synthesis (large chunks often help)
- A made-up phrase not in the doc (e.g. `"submarine sandwich recipe"`) — see what 'closest' looks like when nothing is actually relevant

**Patterns you'll likely see:**
- **Small (300/50):** top hits are tightly on-topic but each chunk is a fragment. If the answer needs facts from two parts of a paragraph, you'd need top-K bigger to recover both.
- **Baseline (1000/200):** top hits feel like "a coherent passage about the topic". Usually the right default.
- **Large (1500/300):** scores are often *lower* (vectors more diluted) and chunks contain a lot of off-topic surrounding text. But the relevant content, when it appears, is more self-contained.

**The concept that makes this click:** chunk size = the resolution at which retrieval can find things. You're trading **focus** (small) vs **completeness** (large).

## Caveats / what we skipped (so you don't over-generalize)

- One doc, one question per run — not a benchmark, just intuition.
- No re-ranker — that often rescues small-chunk recall by widening top-K then re-ordering.
- No metadata filter — in FSR the *first* step is filter by ESN/serial; we'd embed-search within that subset.
- No query rewriting / multi-query — production RAG often expands a single user question into 3–5 variations, retrieves for each, and merges.
- `cl100k_base` token counts are an approximation for the Azure embedding model (different tokenizer, similar order of magnitude).
- FAISS Flat = exhaustive scan, fine for hundreds of chunks; for millions you'd use `IndexIVFFlat` / `IndexHNSW` (cert vocab note).